# Build hourly online features with missing PM as NaN

This notebook rebuilds the online streaming CSV from the full weather grid and raw Pulse Eco measurements.

Goal:
- keep one hourly row per sensor/timestamp from the weather grid
- attach observed PM10/PM2.5 where Pulse Eco has data
- leave missing PM10/PM2.5 as blank CSV cells, which pandas reads back as NaN
- avoid interpolation, ffill, bfill, or median filling

In [23]:
from pathlib import Path

import pandas as pd

In [24]:
# Paths
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "feature_engineering":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_WEATHER_CSV = PROJECT_ROOT / "data" / "streaming" / "bitola_sensor_weather_features_online.csv"
PULSE_DIR = PROJECT_ROOT / "feature_engineering" / "pulse_data"

# Keep the same sensor-quality rule as the observed-only CSV.
# Set to None if you want to keep every sensor, including sensors with 100% missing PM values.
MAX_MISSING_PERCENT = 50.0

# Optional: fill only short PM gaps after joining raw Pulse measurements.
# Long outages stay NaN so we do not fabricate weeks of pollution values.
INTERPOLATE_SHORT_GAPS = True
MAX_INTERPOLATION_GAP_HOURS = 6

output_name = (
    "bitola_sensor_weather_features_online_short_gap_interpolated.csv"
    if INTERPOLATE_SHORT_GAPS
    else "bitola_sensor_weather_features_online_hourly_nan.csv"
)
OUTPUT_CSV = PROJECT_ROOT / "data" / "streaming" / output_name

SOURCE_WEATHER_CSV, PULSE_DIR, OUTPUT_CSV

(PosixPath('/mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/streaming/bitola_sensor_weather_features_online.csv'),
 PosixPath('/mnt/c/Users/RazorVision/Desktop/project-vrnmp/feature_engineering/pulse_data'),
 PosixPath('/mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/streaming/bitola_sensor_weather_features_online_short_gap_interpolated.csv'))

## 1. Load the weather grid

The original online CSV already has the hourly weather rows we need. We drop the old PM columns because those were filled/imputed before. Then we join raw Pulse PM values again.

In [25]:
weather = pd.read_csv(SOURCE_WEATHER_CSV)
weather = weather.drop(columns=["pm10", "pm25"], errors="ignore")

weather["timestamp"] = pd.to_datetime(weather["timestamp"], utc=True, errors="coerce").dt.floor("h")
weather["sensorId"] = weather["sensorId"].astype(str)

weather = (
    weather
    .dropna(subset=["timestamp", "sensorId"])
    .sort_values(["sensorId", "timestamp"])
    .drop_duplicates(subset=["sensorId", "timestamp"], keep="last")
    .reset_index(drop=True)
)

print(f"Weather grid rows: {len(weather):,}")
print(f"Sensors in weather grid: {weather['sensorId'].nunique():,}")
print(f"Date range: {weather['timestamp'].min()} -> {weather['timestamp'].max()}")
weather.head()

Weather grid rows: 48,026
Sensors in weather grid: 22
Date range: 2025-12-01 00:00:00+00:00 -> 2026-03-01 22:00:00+00:00


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.9505,92.41695,4.293669,303.02386,943.6906
1,2025-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.0005,92.41994,3.818376,315.00010,944.0742
2,2025-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.6505,93.06822,3.893995,326.30990,944.1686
3,2025-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4005,93.72927,4.104631,322.12494,944.4741
4,2025-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.3005,94.06355,3.563818,315.00010,944.8185


## 2. Load raw Pulse PM measurements

Pulse timestamps are converted to UTC and floored to the hour so they align with the weather grid. If multiple readings land in the same sensor/hour/type, we use the mean.

In [26]:
pulse_files = sorted(PULSE_DIR.rglob("*.csv"))
print(f"Pulse files found: {len(pulse_files):,}")

pulse_parts = []
for file_path in pulse_files:
    part = pd.read_csv(file_path)
    part["source_file"] = str(file_path.relative_to(PROJECT_ROOT))
    pulse_parts.append(part)

pulse = pd.concat(pulse_parts, ignore_index=True)
pulse = pulse[pulse["type"].isin(["pm10", "pm25"])].copy()

pulse["timestamp"] = pd.to_datetime(pulse["timestamp"], utc=True, errors="coerce").dt.floor("h")
pulse["sensorId"] = pulse["sensorId"].astype(str)
pulse["value"] = pd.to_numeric(pulse["value"], errors="coerce")

pulse = pulse.dropna(subset=["timestamp", "sensorId", "type", "value"])

hourly_pm = (
    pulse
    .groupby(["sensorId", "timestamp", "type"], as_index=False)["value"]
    .mean()
    .pivot(index=["sensorId", "timestamp"], columns="type", values="value")
    .reset_index()
    .rename_axis(columns=None)
)

for column in ["pm10", "pm25"]:
    if column not in hourly_pm.columns:
        hourly_pm[column] = pd.NA

hourly_pm = hourly_pm[["sensorId", "timestamp", "pm10", "pm25"]]

print(f"Hourly PM rows: {len(hourly_pm):,}")
print(f"Sensors with any PM data: {hourly_pm['sensorId'].nunique():,}")
hourly_pm.head()

Pulse files found: 286


/tmp/ipykernel_27822/3921039424.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pulse = pd.concat(pulse_parts, ignore_index=True)


Hourly PM rows: 21,585
Sensors with any PM data: 12


,sensorId,timestamp,pm10,pm25
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 00:00:00+00:00,34.75,17.50
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 01:00:00+00:00,18.25,10.00
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 02:00:00+00:00,15.00,8.75
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 03:00:00+00:00,14.75,7.25
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 04:00:00+00:00,19.00,8.75


## 3. Join PM onto the hourly weather grid

This keeps all weather-grid rows. Missing PM stays missing.

In [27]:
merged = weather.merge(
    hourly_pm,
    on=["sensorId", "timestamp"],
    how="left",
)

missing_by_sensor = (
    merged
    .groupby("sensorId")[["pm10", "pm25"]]
    .apply(lambda frame: frame.isna().mean() * 100)
    .round(2)
    .sort_values("pm10", ascending=False)
)

has_pm_data = (
    merged
    .groupby("sensorId")[["pm10", "pm25"]]
    .apply(lambda frame: frame.notna().any())
)

print("Sensors with any PM data:")
print(has_pm_data.sum())

missing_by_sensor

Sensors with any PM data:
pm10    12
pm25    12
dtype: int64


,pm10,pm25
sensorId,,
ece1058a-ecab-4736-872f-790145aaadfe,100.00,100.00
24039f11-a4fc-4b2d-8bc0-6fd36059f117,100.00,100.00
30dab8a6-ff63-43ce-9a3b-99f1f3f7054d,100.00,100.00
e20e9778-a020-4b86-932a-b7ab6a713a00,100.00,100.00
692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,100.00,100.00
d851c0b9-990e-41db-9c53-529f88524cf9,100.00,100.00
a17013e7-8d1d-4b0d-8e2f-e0881dbca3ac,100.00,100.00
a9a2083f-f086-4fae-bdae-355b391f436b,100.00,100.00
be427cee-4c3a-4aa2-a1ce-9795a74533be,100.00,100.00


## 4. Optionally drop sensors with too much missing PM data

By default this keeps only sensors where both PM10 and PM2.5 missingness is <= 50%.

Unlike the previous observed-only CSV, this does not drop individual missing rows. It keeps the hourly rows and leaves PM as NaN.

In [28]:
if MAX_MISSING_PERCENT is None:
    kept_sensors = missing_by_sensor.index
else:
    kept_sensors = missing_by_sensor[
        (missing_by_sensor["pm10"] <= MAX_MISSING_PERCENT)
        & (missing_by_sensor["pm25"] <= MAX_MISSING_PERCENT)
    ].index

hourly_nan = (
    merged[merged["sensorId"].isin(kept_sensors)]
    .sort_values(["timestamp", "sensorId"])
    .reset_index(drop=True)
)

print(f"Kept sensors: {len(kept_sensors):,} / {missing_by_sensor.shape[0]:,}")
print(f"Output rows: {len(hourly_nan):,}")
print("Missing PM percent in output:")
print((hourly_nan[["pm10", "pm25"]].isna().mean() * 100).round(2))

hourly_nan.head()

Kept sensors: 10 / 22
Output rows: 21,830
Missing PM percent in output:
pm10    6.79
pm25    6.78
dtype: float64


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
0,2025-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.9505,92.41695,4.293669,303.02386,943.69060,34.75,17.50
1,2025-12-01 00:00:00+00:00,2002,41.030221,21.336733,1.9635,92.41773,4.293669,303.02386,943.92490,115.00,103.00
2,2025-12-01 00:00:00+00:00,23b735ef-a996-4a7f-9998-2aa7e78827b0,41.050417,21.346658,2.0675,92.42392,4.293669,303.02386,945.80180,93.75,41.25
3,2025-12-01 00:00:00+00:00,40f081a6-4095-43f7-bffb-64e2af8c026e,41.040130,21.340139,2.0850,100.00000,6.681856,265.36462,939.46530,56.75,34.75
4,2025-12-01 00:00:00+00:00,7b316592-8036-41e2-b8dc-b06b6a9afd54,41.038285,21.327864,2.0330,100.00000,6.681856,265.36462,938.53284,93.50,56.50


## 5. Optionally interpolate short PM gaps

When enabled, this fills only short consecutive missing PM gaps per sensor. Long outages remain NaN.

In [29]:
def missing_pm_summary(df):
    return pd.DataFrame(
        {
            "missing_count": df[["pm10", "pm25"]].isna().sum(),
            "missing_percent": (df[["pm10", "pm25"]].isna().mean() * 100).round(2),
        }
    )


def missing_pm_by_sensor(df):
    return (
        df
        .groupby("sensorId")[["pm10", "pm25"]]
        .apply(lambda frame: frame.isna().mean() * 100)
        .round(2)
        .sort_values("pm10", ascending=False)
    )


def interpolate_short_pm_gaps(df, value_columns=("pm10", "pm25"), max_gap_hours=6):
    result = df.sort_values(["sensorId", "timestamp"]).copy()

    for column in value_columns:
        result[column] = (
            result
            .groupby("sensorId", group_keys=False)[column]
            .apply(
                lambda series: series.interpolate(
                    method="linear",
                    limit=max_gap_hours,
                    limit_area="inside",
                )
            )
        )

    return result

missing_before_interpolation = missing_pm_summary(hourly_nan)
missing_by_sensor_before_interpolation = missing_pm_by_sensor(hourly_nan)

if INTERPOLATE_SHORT_GAPS:
    hourly_nan = interpolate_short_pm_gaps(
        hourly_nan,
        max_gap_hours=MAX_INTERPOLATION_GAP_HOURS,
    )
    print(f"Interpolated PM gaps up to {MAX_INTERPOLATION_GAP_HOURS} consecutive hours.")
else:
    print("Short-gap interpolation disabled. Missing PM values remain NaN.")

missing_after_interpolation = missing_pm_summary(hourly_nan)
missing_by_sensor_after_interpolation = missing_pm_by_sensor(hourly_nan)

missing_change = missing_before_interpolation.join(
    missing_after_interpolation,
    lsuffix="_before",
    rsuffix="_after",
)
missing_change["filled_count"] = (
    missing_change["missing_count_before"] - missing_change["missing_count_after"]
)
missing_change["filled_percent_points"] = (
    missing_change["missing_percent_before"] - missing_change["missing_percent_after"]
).round(2)

missing_by_sensor_change = missing_by_sensor_before_interpolation.join(
    missing_by_sensor_after_interpolation,
    lsuffix="_before",
    rsuffix="_after",
)
missing_by_sensor_change["pm10_filled_percent_points"] = (
    missing_by_sensor_change["pm10_before"] - missing_by_sensor_change["pm10_after"]
).round(2)
missing_by_sensor_change["pm25_filled_percent_points"] = (
    missing_by_sensor_change["pm25_before"] - missing_by_sensor_change["pm25_after"]
).round(2)

print("Overall missingness before/after interpolation:")
display(missing_change)

print("Missingness by sensor before/after interpolation:")
missing_by_sensor_change

Interpolated PM gaps up to 6 consecutive hours.
Overall missingness before/after interpolation:


,missing_count_before,missing_percent_before,missing_count_after,missing_percent_after,filled_count,filled_percent_points
pm10,1482,6.79,1130,5.18,352,1.61
pm25,1480,6.78,1130,5.18,350,1.60


Missingness by sensor before/after interpolation:


,pm10_before,pm25_before,pm10_after,pm25_after,pm10_filled_percent_points,pm25_filled_percent_points
sensorId,,,,,,
2002,42.69,42.60,38.71,38.71,3.98,3.89
fec52a19-9148-4350-a1b4-ae0da05ee199,7.79,7.79,4.54,4.54,3.25,3.25
d241a044-0a06-40c2-9d90-c91fd0a95060,7.24,7.24,6.05,6.05,1.19,1.19
874ff9c6-786d-45fc-a90e-48c7ffe03417,4.81,4.81,1.10,1.10,3.71,3.71
16836a55-7140-43e2-9a63-56fac5cba714,1.88,1.88,0.87,0.87,1.01,1.01
7b316592-8036-41e2-b8dc-b06b6a9afd54,1.60,1.60,0.14,0.14,1.46,1.46
87f82783-853b-417d-8964-b5cf11e44873,0.73,0.73,0.09,0.09,0.64,0.64
23b735ef-a996-4a7f-9998-2aa7e78827b0,0.41,0.41,0.09,0.09,0.32,0.32
40f081a6-4095-43f7-bffb-64e2af8c026e,0.37,0.37,0.09,0.09,0.28,0.28


In [30]:
# filter to show only nan values for pm10 and pm25
hourly_nan[hourly_nan[["pm10", "pm25"]].isna().any(axis=1)]


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
2470,2025-12-11 07:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,3.5505,75.917920,6.989936,325.49140,949.94385,NaN,NaN
2480,2025-12-11 08:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,6.2005,71.100790,8.217153,331.18930,950.43630,NaN,NaN
15130,2026-02-02 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.4505,83.566770,9.044888,328.84076,932.97705,NaN,NaN
15140,2026-02-02 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4005,89.095604,7.841887,328.13406,932.70760,NaN,NaN
15150,2026-02-02 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4005,88.772530,5.751556,339.86360,932.80020,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
20189,2026-02-23 02:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,-0.6110,81.954520,4.236697,257.73523,946.30365,NaN,NaN
20199,2026-02-23 03:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,-0.8610,85.665830,4.327493,253.07240,945.77330,NaN,NaN
20209,2026-02-23 04:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,-1.4110,90.519600,4.445672,248.62930,945.34530,NaN,NaN
20219,2026-02-23 05:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,-1.5110,90.848724,4.589118,244.44008,945.04070,NaN,NaN


## 6. Save CSV

Missing PM values are written as blank cells. When you load this CSV with pandas later, those blanks become NaN.

In [31]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
hourly_nan.to_csv(OUTPUT_CSV, index=False, na_rep="")

print(f"Saved: {OUTPUT_CSV}")

Saved: /mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/streaming/bitola_sensor_weather_features_online_short_gap_interpolated.csv


## 7. Verify that blanks reload as NaN

In [32]:
check = pd.read_csv(OUTPUT_CSV)

print(f"Reloaded rows: {len(check):,}")
print("Missing values after reloading:")
print(check[["pm10", "pm25"]].isna().sum())
print("Missing percent after reloading:")
print((check[["pm10", "pm25"]].isna().mean() * 100).round(2))

check[check[["pm10", "pm25"]].isna().any(axis=1)].head(20)

Reloaded rows: 21,830
Missing values after reloading:
pm10    1130
pm25    1130
dtype: int64
Missing percent after reloading:
pm10    5.18
pm25    5.18
dtype: float64


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
247,2025-12-11 07:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,3.5505,75.917920,6.989936,325.491400,949.94385,NaN,NaN
248,2025-12-11 08:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,6.2005,71.100790,8.217153,331.189300,950.43630,NaN,NaN
1513,2026-02-02 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.4505,83.566770,9.044888,328.840760,932.97705,NaN,NaN
1514,2026-02-02 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4005,89.095604,7.841887,328.134060,932.70760,NaN,NaN
1515,2026-02-02 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4005,88.772530,5.751556,339.863600,932.80020,NaN,NaN
1516,2026-02-02 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4505,88.134080,6.915374,321.340180,933.09110,NaN,NaN
1517,2026-02-02 05:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.7005,84.392740,7.647352,333.435030,933.61850,NaN,NaN
1518,2026-02-02 06:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.0005,86.915985,5.157558,330.751280,934.15900,NaN,NaN
1519,2026-02-02 07:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.4505,80.290695,8.825508,348.231720,934.83057,NaN,NaN
1520,2026-02-02 08:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.8505,76.353830,8.117980,356.186000,935.48895,NaN,NaN
